# Boundary Conditions: Mathematical Constraints and Physical Reality

## Introduction: Why Boundary Conditions Matter

Imagine solving a heat transfer problem without knowing the temperature at the edges, or a structural problem without knowing how the structure is supported. These are **ill-posed problems** - they have infinitely many solutions or no solution at all.

Boundary conditions (BCs) are the mathematical representation of physical constraints:
- **Temperature at boundaries** (Dirichlet BCs)
- **Heat flux through boundaries** (Neumann BCs)
- **Heat transfer with surrounding environment** (Robin BCs)
- **Connected periodic boundaries** (Periodic BCs)

In FEA, boundary conditions:
1. **Make problems well-posed** (ensure unique solutions)
2. **Represent physical reality** (how the system interacts with environment)
3. **Affect solution accuracy** (proper implementation is crucial)
4. **Influence matrix structure** (affect computational efficiency)

### Learning Objectives

By the end of this notebook, you will:
- Understand the mathematical foundation of different boundary condition types
- Implement Dirichlet, Neumann, and Robin boundary conditions in FEA
- Understand how BCs affect the weak form and matrix assembly
- Handle complex boundary conditions on irregular geometries
- Appreciate the physical interpretation of each BC type

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.tri import Triangulation
from mpl_toolkits.mplot3d import Axes3D
from scipy.spatial import Delaunay
from scipy.sparse import lil_matrix
from scipy.sparse.linalg import spsolve
from matplotlib.patches import Polygon
from matplotlib.collections import PatchCollection

# Set up nice plotting parameters
plt.rcParams['figure.figsize'] = (12, 8)
plt.rcParams['font.size'] = 12
np.set_printoptions(precision=4, suppress=True)

## 1. Mathematical Foundation: Strong vs Weak Form

### The Strong Form

Consider the Poisson equation in domain $\Omega$:

$$-\nabla^2 u = f \text{ in } \Omega$$

This equation alone is not enough - we need boundary conditions. The boundary $\partial\Omega$ can be divided into:
- $\Gamma_D$: Dirichlet boundary (essential BC)
- $\Gamma_N$: Neumann boundary (natural BC)
- $\Gamma_R$: Robin boundary (mixed BC)

The strong form specifies:
$$u = g \text{ on } \Gamma_D \quad \text{(Dirichlet)}$$
$$\frac{\partial u}{\partial n} = h \text{ on } \Gamma_N \quad \text{(Neumann)}$$

### The Weak Form

In FEA, we work with the weak form. Using integration by parts:

$$\int_\Omega \nabla u \cdot \nabla v \, d\Omega = \int_\Omega f v \, d\Omega + \int_{\partial\Omega} \frac{\partial u}{\partial n} v \, ds$$

The boundary integral reveals how different BCs affect the formulation!

In [ ]:
def visualize_boundary_conditions():
    """
    Visualize different types of boundary conditions on a simple domain.
    """
    # Create a rectangular domain
    width, height = 2.0, 1.0
    
    fig, axes = plt.subplots(2, 2, figsize=(15, 10))
    
    # Common setup
    x = np.linspace(0, width, 50)
    y = np.linspace(0, height, 25)
    X, Y = np.meshgrid(x, y)
    
    # 1. Dirichlet Boundary Conditions
    ax1 = axes[0, 0]
    ax1.contourf(X, Y, X, levels=20, cmap='coolwarm', alpha=0.7)
    ax1.plot([0, width, width, 0, 0], [0, 0, height, height, 0], 'k-', linewidth=3)
    
    # Add arrows showing prescribed values
    ax1.arrow(0.1, height/2, 0, 0, head_width=0.05, head_length=0.02, fc='red', ec='red')
    ax1.arrow(width-0.1, height/2, 0, 0, head_width=0.05, head_length=0.02, fc='red', ec='red')
    ax1.arrow(width/2, 0.1, 0, 0, head_width=0.05, head_length=0.02, fc='blue', ec='blue')
    ax1.arrow(width/2, height-0.1, 0, 0, head_width=0.05, head_length=0.02, fc='blue', ec='blue')
    
    ax1.text(width/2, height/2, 'Dirichlet BC\nu = prescribed', 
            ha='center', va='center', fontsize=12, fontweight='bold',
            bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))
    
    ax1.set_title('Dirichlet Boundary Conditions\n(Essential BC)', fontsize=14)
    ax1.set_aspect('equal')
    ax1.set_xlabel('x')
    ax1.set_ylabel('y')
    
    # 2. Neumann Boundary Conditions
    ax2 = axes[0, 1]
    ax2.contourf(X, Y, Y, levels=20, cmap='viridis', alpha=0.7)
    ax2.plot([0, width, width, 0, 0], [0, 0, height, height, 0], 'k-', linewidth=3)
    
    # Add arrows showing flux
    for i in range(5):
        x_pos = (i + 1) * width / 6
        ax2.arrow(x_pos, height, 0, -0.1, head_width=0.03, head_length=0.02, 
                 fc='red', ec='red', alpha=0.7)
    
    ax2.text(width/2, height/2, 'Neumann BC\n∂u/∂n = prescribed flux', 
            ha='center', va='center', fontsize=12, fontweight='bold',
            bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))
    
    ax2.set_title('Neumann Boundary Conditions\n(Natural BC)', fontsize=14)
    ax2.set_aspect('equal')
    ax2.set_xlabel('x')
    ax2.set_ylabel('y')
    
    # 3. Robin Boundary Conditions
    ax3 = axes[1, 0]
    # Create a mixed field
    Z = X + 0.5 * Y
    ax3.contourf(X, Y, Z, levels=20, cmap='plasma', alpha=0.7)
    ax3.plot([0, width, width, 0, 0], [0, 0, height, height, 0], 'k-', linewidth=3)
    
    # Add arrows showing both value and flux
    ax3.arrow(0.1, height/2, 0.1, 0, head_width=0.05, head_length=0.02, fc='blue', ec='blue')
    ax3.arrow(width/2, height-0.1, 0, -0.1, head_width=0.05, head_length=0.02, fc='red', ec='red')
    
    ax3.text(width/2, height/2, 'Robin BC\nαu + β∂u/∂n = prescribed', 
            ha='center', va='center', fontsize=11, fontweight='bold',
            bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))
    
    ax3.set_title('Robin Boundary Conditions\n(Mixed BC)', fontsize=14)
    ax3.set_aspect('equal')
    ax3.set_xlabel('x')
    ax3.set_ylabel('y')
    
    # 4. Periodic Boundary Conditions
    ax4 = axes[1, 1]
    # Create a periodic field
    Z_periodic = np.sin(2 * np.pi * X / width) * np.cos(np.pi * Y / height)
    ax4.contourf(X, Y, Z_periodic, levels=20, cmap='cividis', alpha=0.7)
    ax4.plot([0, width, width, 0, 0], [0, 0, height, height, 0], 'k-', linewidth=3)
    
    # Add arrows showing periodic connection
    for i in range(3):
        y_pos = (i + 1) * height / 4
        ax4.annotate('', xy=(0.05, y_pos), xytext=(width-0.05, y_pos),
                    arrowprops=dict(arrowstyle='<->', color='red', lw=2))
    
    ax4.text(width/2, height/2, 'Periodic BC\nu(x=0) = u(x=L)', 
            ha='center', va='center', fontsize=12, fontweight='bold',
            bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))
    
    ax4.set_title('Periodic Boundary Conditions\n(Connected boundaries)', fontsize=14)
    ax4.set_aspect('equal')
    ax4.set_xlabel('x')
    ax4.set_ylabel('y')
    
    plt.tight_layout()
    plt.show()
    
    print("Boundary Condition Types:")
    print("\n1. Dirichlet (Essential):")
    print("   - Prescribes the value of u at boundary")
    print("   - Applied by modifying the linear system")
    print("   - Example: Temperature, displacement")
    
    print("\n2. Neumann (Natural):")
    print("   - Prescribes the normal derivative ∂u/∂n")
    print("   - Appears naturally in weak form")
    print("   - Example: Heat flux, traction")
    
    print("\n3. Robin (Mixed):")
    print("   - Combines value and derivative")
    print("   - Models convective heat transfer")
    print("   - Example: Convection, springs")
    
    print("\n4. Periodic:")
    print("   - Connects opposite boundaries")
    print("   - Used for repeating structures")
    print("   - Example: Unit cells, crystals")

visualize_boundary_conditions()

## 2. Dirichlet Boundary Conditions: Essential Constraints

### Mathematical Formulation

Dirichlet boundary conditions specify the value of the solution:

$$u = g \text{ on } \Gamma_D$$

In the weak form, Dirichlet BCs constrain the **trial and test function spaces**:
- Trial functions: $v \in V$ where $v = g$ on $\Gamma_D$
- Test functions: $w \in V_0$ where $w = 0$ on $\Gamma_D$

### Implementation Strategies

Let's explore different ways to implement Dirichlet BCs in FEA.

In [ ]:
def dirichlet_implementation_methods():
    """
    Compare different methods for implementing Dirichlet boundary conditions.
    """
    def create_test_problem():
        """
        Create a simple 1D Poisson problem for testing.
        """
        # 1D problem: -u'' = 1, u(0) = 0, u(1) = 1
        n_nodes = 6
        nodes = np.linspace(0, 1, n_nodes)
        elements = np.array([[i, i+1] for i in range(n_nodes-1)])
        
        # Assemble system
        n_nodes = len(nodes)
        K = lil_matrix((n_nodes, n_nodes))
        f = np.zeros(n_nodes)
        
        for elem in elements:
            node1, node2 = elem
            x1, x2 = nodes[node1], nodes[node2]
            h = x2 - x1
            
            # Element stiffness matrix
            Ke = (1/h) * np.array([[1, -1], [-1, 1]])
            # Element load vector
            fe = np.array([h/2, h/2])
            
            # Assembly
            for i in range(2):
                for j in range(2):
                    K[elem[i], elem[j]] += Ke[i, j]
                f[elem[i]] += fe[i]
        
        return nodes, K.tocsr(), f, [0, n_nodes-1]  # boundary nodes
    
    def method1_penalty(K, f, boundary_nodes, values, penalty=1e10):
        """
        Method 1: Penalty method.
        """
        K_mod = K.copy()
        f_mod = f.copy()
        
        for node, value in zip(boundary_nodes, values):
            K_mod[node, node] += penalty
            f_mod[node] += penalty * value
        
        return K_mod, f_mod
    
    def method2_elimination(K, f, boundary_nodes, values):
        """
        Method 2: Row elimination method.
        """
        K_mod = K.copy()
        f_mod = f.copy()
        
        for node, value in zip(boundary_nodes, values):
            # Set row to identity
            K_mod[node, :] = 0
            K_mod[node, node] = 1
            f_mod[node] = value
            
            # Modify other equations
            for i in range(K.shape[0]):
                if i != node:
                    f_mod[i] -= K[i, node] * value
                    K_mod[i, node] = 0
        
        return K_mod, f_mod
    
    def method3_submatrix(K, f, boundary_nodes, values):
        """
        Method 3: Submatrix method (exact).
        """
        n_nodes = K.shape[0]
        free_nodes = [i for i in range(n_nodes) if i not in boundary_nodes]
        
        # Extract submatrix for free nodes
        K_reduced = K[free_nodes][:, free_nodes].tocsr()
        f_reduced = f.copy()
        
        # Modify RHS for boundary conditions
        for node, value in zip(boundary_nodes, values):
            for i, free_node in enumerate(free_nodes):
                f_reduced[free_node] -= K[free_node, node] * value
        
        f_reduced = f_reduced[free_nodes]
        
        return K_reduced, f_reduced, free_nodes
    
    # Test all methods
    nodes, K, f, boundary_nodes = create_test_problem()
    bc_values = [0, 1]  # u(0) = 0, u(1) = 1
    
    # Method 1: Penalty
    K1, f1 = method1_penalty(K, f, boundary_nodes, bc_values)
    u1 = spsolve(K1, f1)
    
    # Method 2: Elimination
    K2, f2 = method2_elimination(K, f, boundary_nodes, bc_values)
    u2 = spsolve(K2, f2)
    
    # Method 3: Submatrix
    K3, f3, free_nodes = method3_submatrix(K, f, boundary_nodes, bc_values)
    u3_free = spsolve(K3, f3)
    u3 = np.zeros(len(nodes))
    u3[free_nodes] = u3_free
    for node, value in zip(boundary_nodes, bc_values):
        u3[node] = value
    
    # Analytical solution: u(x) = x + 0.5*x*(1-x)
    u_analytical = nodes + 0.5 * nodes * (1 - nodes)
    
    # Visualize results
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))
    
    # Plot solutions
    ax1.plot(nodes, u_analytical, 'k-', linewidth=3, label='Analytical')
    ax1.plot(nodes, u1, 'ro-', linewidth=2, markersize=8, label='Penalty method')
    ax1.plot(nodes, u2, 'bs-', linewidth=2, markersize=8, label='Elimination method')
    ax1.plot(nodes, u3, 'g^-', linewidth=2, markersize=8, label='Submatrix method')
    
    ax1.set_xlabel('x')
    ax1.set_ylabel('u(x)')
    ax1.set_title('Comparison of Dirichlet BC Implementation Methods')
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    
    # Plot errors
    error1 = np.abs(u1 - u_analytical)
    error2 = np.abs(u2 - u_analytical)
    error3 = np.abs(u3 - u_analytical)
    
    ax2.semilogy(nodes, error1, 'ro-', linewidth=2, markersize=8, label='Penalty method')
    ax2.semilogy(nodes, error2, 'bs-', linewidth=2, markersize=8, label='Elimination method')
    ax2.semilogy(nodes, error3, 'g^-', linewidth=2, markersize=8, label='Submatrix method')
    
    ax2.set_xlabel('x')
    ax2.set_ylabel('Absolute Error')
    ax2.set_title('Implementation Error Comparison')
    ax2.legend()
    ax2.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    # Print results table
    print("Dirichlet BC Implementation Comparison:")
    print("=" * 60)
    print(f"{'Node':<6} {'Analytical':<12} {'Penalty':<12} {'Elimination':<12} {'Submatrix':<12}")
    print("-" * 60)
    
    for i in range(len(nodes)):
        print(f"{i:<6} {u_analytical[i]:<12.6f} {u1[i]:<12.6f} {u2[i]:<12.6f} {u3[i]:<12.6f}")
    
    print("\nMethod Characteristics:")
    print("\nPenalty Method:")
    print("- Pros: Easy to implement, preserves matrix structure")
    print("- Cons: Approximate (depends on penalty parameter), ill-conditioning")
    
    print("\nElimination Method:")
    print("- Pros: Exact, stable")
    print("- Cons: Modifies matrix structure, more complex implementation")
    
    print("\nSubmatrix Method:")
    print("- Pros: Most efficient, mathematically exact")
    print("- Cons: Requires bookkeeping for free nodes")
    
    print("\nRecommendation: Use submatrix method for accuracy, penalty method for simplicity.")

dirichlet_implementation_methods()

## 3. Neumann Boundary Conditions: Natural Constraints

### Mathematical Formulation

Neumann boundary conditions specify the normal derivative:

$$\frac{\partial u}{\partial n} = h \text{ on } \Gamma_N$$

In the weak form, Neumann BCs appear **naturally** through the boundary integral:

$$\int_\Omega \nabla u \cdot \nabla v \, d\Omega = \int_\Omega f v \, d\Omega + \int_{\Gamma_N} h v \, ds$$

The key insight: **Neumann BCs contribute to the load vector, not the stiffness matrix!**

### Implementation Strategy

Let's implement Neumann BCs for a 2D heat transfer problem.

In [ ]:
def neumann_implementation_2d():
    """
    Implement Neumann boundary conditions in 2D.
    """
    def create_mesh_and_assemble():
        """
        Create mesh and assemble basic system.
        """
        # Create structured mesh
        nx, ny = 8, 8
        x = np.linspace(0, 1, nx)
        y = np.linspace(0, 1, ny)
        X, Y = np.meshgrid(x, y)
        nodes = np.column_stack([X.ravel(), Y.ravel()])
        
        # Create triangulation
        tri = Delaunay(nodes)
        elements = tri.simplices
        
        # Assemble system
        n_nodes = len(nodes)
        K = lil_matrix((n_nodes, n_nodes))
        f = np.zeros(n_nodes)
        
        for elem in elements:
            coords = nodes[elem]
            x_coords, y_coords = coords[:, 0], coords[:, 1]
            
            # Element area
            A = 0.5 * abs(np.linalg.det(np.array([
                [1, x_coords[0], y_coords[0]],
                [1, x_coords[1], y_coords[1]],
                [1, x_coords[2], y_coords[2]]
            ])))
            
            # Shape function gradients
            b = np.array([y_coords[1] - y_coords[2], y_coords[2] - y_coords[0], y_coords[0] - y_coords[1]])
            c = np.array([x_coords[2] - x_coords[1], x_coords[0] - x_coords[2], x_coords[1] - x_coords[0]])
            B = np.array([b, c]) / (2 * A)
            
            # Element stiffness
            Ke = A * (B.T @ B)
            
            # Element load (constant source)
            fe = np.ones(3) * A / 3.0
            
            # Assembly
            for i in range(3):
                for j in range(3):
                    K[elem[i], elem[j]] += Ke[i, j]
                f[elem[i]] += fe[i]
        
        return nodes, elements, K.tocsr(), f
    
    def identify_boundary_edges(nodes, elements):
        """
        Identify boundary edges for Neumann BC application.
        """
        # Find all edges
        edges = set()
        edge_count = {}
        
        for elem in elements:
            for i in range(3):
                j = (i + 1) % 3
                edge = tuple(sorted([elem[i], elem[j]]))
                edges.add(edge)
                edge_count[edge] = edge_count.get(edge, 0) + 1
        
        # Boundary edges appear only once
        boundary_edges = [edge for edge, count in edge_count.items() if count == 1]
        
        return boundary_edges
    
    def apply_neumann_bc(nodes, elements, K, f, flux_function):
        """
        Apply Neumann boundary conditions using boundary integration.
        """
        f_modified = f.copy()
        
        # Identify boundary edges
        boundary_edges = identify_boundary_edges(nodes, elements)
        
        for edge in boundary_edges:
            node1, node2 = edge
            x1, y1 = nodes[node1]
            x2, y2 = nodes[node2]
            
            # Edge midpoint for flux evaluation
            x_mid = (x1 + x2) / 2
            y_mid = (y1 + y2) / 2
            
            # Edge length
            edge_length = np.sqrt((x2 - x1)**2 + (y2 - y1)**2)
            
            # Evaluate flux at midpoint
            flux = flux_function(x_mid, y_mid)
            
            # Apply Neumann BC: ∫_Γ h*v ds ≈ h * edge_length * (v1 + v2)/2
            # For linear shape functions, this gives equal contribution to both nodes
            contribution = flux * edge_length / 2
            f_modified[node1] += contribution
            f_modified[node2] += contribution
        
        return f_modified
    
    def solve_with_neumann(flux_function, bc_type):
        """
        Solve system with Neumann BCs.
        """
        # Create mesh and assemble
        nodes, elements, K, f = create_mesh_and_assemble()
        
        # Apply Neumann BCs
        f_neumann = apply_neumann_bc(nodes, elements, K, f, flux_function)
        
        # Apply Dirichlet BCs on some boundary (needed for well-posedness)
        boundary_nodes = []
        for i, (x, y) in enumerate(nodes):
            if abs(x) < 1e-10:  # Left boundary
                boundary_nodes.append(i)
        
        # Apply u = 0 on left boundary
        K_bc = K.copy()
        f_bc = f_neumann.copy()
        
        penalty = 1e10
        for node in boundary_nodes:
            K_bc[node, node] += penalty
        
        # Solve
        u = spsolve(K_bc, f_bc)
        
        return nodes, elements, u
    
    # Test different Neumann BCs
    fig, axes = plt.subplots(2, 2, figsize=(15, 12))
    
    # Test 1: Zero flux (insulated)
    def zero_flux(x, y):
        return 0.0
    
    nodes1, elements1, u1 = solve_with_neumann(zero_flux, "zero")
    ax1 = axes[0, 0]
    contour1 = ax1.tricontourf(nodes1[:, 0], nodes1[:, 1], elements1, u1, levels=15, cmap='viridis')
    plt.colorbar(contour1, ax=ax1)
    ax1.set_title('Zero Flux (Insulated Boundary)\n∂u/∂n = 0')
    ax1.set_aspect('equal')
    
    # Test 2: Constant positive flux
    def constant_flux(x, y):
        return 2.0  # Constant heat flux into domain
    
    nodes2, elements2, u2 = solve_with_neumann(constant_flux, "constant")
    ax2 = axes[0, 1]
    contour2 = ax2.tricontourf(nodes2[:, 0], nodes2[:, 1], elements2, u2, levels=15, cmap='viridis')
    plt.colorbar(contour2, ax=ax2)
    ax2.set_title('Constant Positive Flux\n∂u/∂n = 2')
    ax2.set_aspect('equal')
    
    # Test 3: Spatially varying flux
    def varying_flux(x, y):
        return 5 * np.sin(2 * np.pi * y)  # Flux varies with position
    
    nodes3, elements3, u3 = solve_with_neumann(varying_flux, "varying")
    ax3 = axes[1, 0]
    contour3 = ax3.tricontourf(nodes3[:, 0], nodes3[:, 1], elements3, u3, levels=15, cmap='viridis')
    plt.colorbar(contour3, ax=ax3)
    ax3.set_title('Spatially Varying Flux\n∂u/∂n = 5sin(2πy)')
    ax3.set_aspect('equal')
    
    # Test 4: Flux only on specific boundary
    def top_flux_only(x, y):
        if abs(y - 1.0) < 0.1:  # Top boundary
            return 3.0
        return 0.0  # No flux elsewhere
    
    nodes4, elements4, u4 = solve_with_neumann(top_flux_only, "selective")
    ax4 = axes[1, 1]
    contour4 = ax4.tricontourf(nodes4[:, 0], nodes4[:, 1], elements4, u4, levels=15, cmap='viridis')
    plt.colorbar(contour4, ax=ax4)
    ax4.set_title('Selective Flux (Top Boundary Only)\n∂u/∂n = 3 on y=1')
    ax4.set_aspect('equal')
    
    plt.tight_layout()
    plt.show()
    
    print("Neumann Boundary Conditions Implementation:")
    print("=" * 50)
    print("\nKey Insights:")
    print("1. Neumann BCs contribute to the load vector, not stiffness matrix")
    print("2. Implementation involves boundary integration")
    print("3. Requires at least one Dirichlet BC for well-posedness")
    print("4. Natural in weak form - no special treatment needed")
    
    print("\nPhysical Interpretation:")
    print("- Zero flux: Insulated boundary (no heat transfer)")
    print("- Positive flux: Heat/flux entering domain")
    print("- Negative flux: Heat/flux leaving domain")
    print("- Varying flux: Non-uniform boundary conditions")
    
    print("\nImplementation Notes:")
    print("- Use boundary edge detection")
    print("- Numerical integration: flux × edge length")
    print("- Equal contribution to edge nodes for linear elements")

neumann_implementation_2d()

## 4. Robin Boundary Conditions: Mixed Constraints

### Mathematical Formulation

Robin boundary conditions combine value and derivative:

$$\alpha u + \beta \frac{\partial u}{\partial n} = g \text{ on } \Gamma_R$$

A common form is the convective heat transfer condition:

$$-k \frac{\partial u}{\partial n} = h(u - u_\infty)$$

or rearranged:

$$\frac{\partial u}{\partial n} = -\frac{h}{k}(u - u_\infty)$$

### Implementation in FEA

Robin BCs contribute to **both** the stiffness matrix and load vector!

In [ ]:
def robin_implementation():
    """
    Implement Robin boundary conditions for convective heat transfer.
    """
    def create_mesh_and_assemble():
        """
        Create mesh and assemble basic system.
        """
        # Create structured mesh
        nx, ny = 10, 10
        x = np.linspace(0, 1, nx)
        y = np.linspace(0, 1, ny)
        X, Y = np.meshgrid(x, y)
        nodes = np.column_stack([X.ravel(), Y.ravel()])
        
        # Create triangulation
        tri = Delaunay(nodes)
        elements = tri.simplices
        
        # Assemble system
        n_nodes = len(nodes)
        K = lil_matrix((n_nodes, n_nodes))
        f = np.zeros(n_nodes)
        
        for elem in elements:
            coords = nodes[elem]
            x_coords, y_coords = coords[:, 0], coords[:, 1]
            
            # Element area
            A = 0.5 * abs(np.linalg.det(np.array([
                [1, x_coords[0], y_coords[0]],
                [1, x_coords[1], y_coords[1]],
                [1, x_coords[2], y_coords[2]]
            ])))
            
            # Shape function gradients
            b = np.array([y_coords[1] - y_coords[2], y_coords[2] - y_coords[0], y_coords[0] - y_coords[1]])
            c = np.array([x_coords[2] - x_coords[1], x_coords[0] - x_coords[2], x_coords[1] - x_coords[0]])
            B = np.array([b, c]) / (2 * A)
            
            # Element stiffness
            Ke = A * (B.T @ B)
            
            # Element load (heat source)
            fe = np.ones(3) * A * 10  # Heat source term
            
            # Assembly
            for i in range(3):
                for j in range(3):
                    K[elem[i], elem[j]] += Ke[i, j]
                f[elem[i]] += fe[i]
        
        return nodes, elements, K.tocsr(), f
    
    def identify_boundary_edges(nodes, elements):
        """
        Identify boundary edges.
        """
        edges = set()
        edge_count = {}
        
        for elem in elements:
            for i in range(3):
                j = (i + 1) % 3
                edge = tuple(sorted([elem[i], elem[j]]))
                edges.add(edge)
                edge_count[edge] = edge_count.get(edge, 0) + 1
        
        boundary_edges = [edge for edge, count in edge_count.items() if count == 1]
        return boundary_edges
    
    def apply_robin_bc(nodes, K, f, h_conv, k_thermal, T_infinity, boundary_selector):
        """
        Apply Robin BC: -k∂u/∂n = h(u - T_∞)
        """
        K_modified = K.copy()
        f_modified = f.copy()
        
        # Get elements to find boundary edges
        nx, ny = 10, 10
        x = np.linspace(0, 1, nx)
        y = np.linspace(0, 1, ny)
        X, Y = np.meshgrid(x, y)
        mesh_nodes = np.column_stack([X.ravel(), Y.ravel()])
        tri = Delaunay(mesh_nodes)
        elements = tri.simplices
        
        boundary_edges = identify_boundary_edges(mesh_nodes, elements)
        
        for edge in boundary_edges:
            node1, node2 = edge
            x1, y1 = mesh_nodes[node1]
            x2, y2 = mesh_nodes[node2]
            
            # Edge midpoint
            x_mid = (x1 + x2) / 2
            y_mid = (y1 + y2) / 2
            
            # Check if this edge should have Robin BC
            if boundary_selector(x_mid, y_mid):
                # Edge length
                edge_length = np.sqrt((x2 - x1)**2 + (y2 - y1)**2)
                
                # Robin BC contribution
                # Stiffness: (h/k) * edge_length * [[1/3, 1/6], [1/6, 1/3]]
                robin_coeff = (h_conv / k_thermal) * edge_length
                Ke_robin = robin_coeff * np.array([[1/3, 1/6], [1/6, 1/3]])
                
                # Load: (h/k) * T_∞ * edge_length * [1/2, 1/2]
                fe_robin = robin_coeff * T_infinity * np.array([1.5, 1.5])
                
                # Add to global system
                local_nodes = [node1, node2]
                for i in range(2):
                    for j in range(2):
                        K_modified[local_nodes[i], local_nodes[j]] += Ke_robin[i, j]
                    f_modified[local_nodes[i]] += fe_robin[i]
        
        return K_modified, f_modified
    
    # Test different Robin BC configurations
    fig, axes = plt.subplots(2, 2, figsize=(15, 12))
    
    # Configuration 1: Convection on right boundary
    def right_boundary(x, y):
        return abs(x - 1.0) < 0.1
    
    nodes, elements, K, f = create_mesh_and_assemble()
    K1, f1 = apply_robin_bc(nodes, K, f, h_conv=10, k_thermal=1, T_infinity=0, boundary_selector=right_boundary)
    
    # Apply Dirichlet BC on left boundary
    boundary_nodes = []
    for i, (x, y) in enumerate(nodes):
        if abs(x) < 1e-10:
            boundary_nodes.append(i)
    
    K1_bc = K1.copy()
    f1_bc = f1.copy()
    penalty = 1e10
    for node in boundary_nodes:
        K1_bc[node, node] += penalty
    
    u1 = spsolve(K1_bc, f1_bc)
    
    ax1 = axes[0, 0]
    contour1 = ax1.tricontourf(nodes[:, 0], nodes[:, 1], elements, u1, levels=15, cmap='coolwarm')
    plt.colorbar(contour1, ax=ax1)
    ax1.set_title('Convection on Right Boundary\nh=10, T∞=0, k=1')
    ax1.set_aspect('equal')
    
    # Configuration 2: Convection on top and bottom
    def top_bottom_boundary(x, y):
        return abs(y - 1.0) < 0.1 or abs(y) < 0.1
    
    K2, f2 = apply_robin_bc(nodes, K, f, h_conv=5, k_thermal=1, T_infinity=20, boundary_selector=top_bottom_boundary)
    
    K2_bc = K2.copy()
    f2_bc = f2.copy()
    for node in boundary_nodes:
        K2_bc[node, node] += penalty
    
    u2 = spsolve(K2_bc, f2_bc)
    
    ax2 = axes[0, 1]
    contour2 = ax2.tricontourf(nodes[:, 0], nodes[:, 1], elements, u2, levels=15, cmap='coolwarm')
    plt.colorbar(contour2, ax=ax2)
    ax2.set_title('Convection on Top/Bottom\nh=5, T∞=20, k=1')
    ax2.set_aspect('equal')
    
    # Configuration 3: Strong convection
    def right_boundary_strong(x, y):
        return abs(x - 1.0) < 0.1
    
    K3, f3 = apply_robin_bc(nodes, K, f, h_conv=50, k_thermal=1, T_infinity=0, boundary_selector=right_boundary_strong)
    
    K3_bc = K3.copy()
    f3_bc = f3.copy()
    for node in boundary_nodes:
        K3_bc[node, node] += penalty
    
    u3 = spsolve(K3_bc, f3_bc)
    
    ax3 = axes[1, 0]
    contour3 = ax3.tricontourf(nodes[:, 0], nodes[:, 1], elements, u3, levels=15, cmap='coolwarm')
    plt.colorbar(contour3, ax=ax3)
    ax3.set_title('Strong Convection on Right\nh=50, T∞=0, k=1')
    ax3.set_aspect('equal')
    
    # Configuration 4: Mixed Robin conditions
    def mixed_boundary(x, y):
        return abs(x - 1.0) < 0.1 or (abs(y - 1.0) < 0.1 and x > 0.5)
    
    K4, f4 = apply_robin_bc(nodes, K, f, h_conv=15, k_thermal=2, T_infinity=25, boundary_selector=mixed_boundary)
    
    K4_bc = K4.copy()
    f4_bc = f4.copy()
    for node in boundary_nodes:
        K4_bc[node, node] += penalty
    
    u4 = spsolve(K4_bc, f4_bc)
    
    ax4 = axes[1, 1]
    contour4 = ax4.tricontourf(nodes[:, 0], nodes[:, 1], elements, u4, levels=15, cmap='coolwarm')
    plt.colorbar(contour4, ax=ax4)
    ax4.set_title('Mixed Robin Conditions\nh=15, T∞=25, k=2')
    ax4.set_aspect('equal')
    
    plt.tight_layout()
    plt.show()
    
    print("Robin Boundary Conditions Implementation:")
    print("=" * 50)
    print("\nMathematical Form:")
    print("- General: αu + β∂u/∂n = g")
    print("- Convection: -k∂u/∂n = h(u - T∞)")
    print("- Rearranged: ∂u/∂n = -(h/k)(u - T∞)")
    
    print("\nFEA Implementation:")
    print("1. Contributes to BOTH stiffness matrix and load vector")
    print("2. Stiffness contribution: (h/k) × boundary integral")
    print("3. Load contribution: (h/k) × T∞ × boundary integral")
    print("4. Uses edge-based boundary integration")
    
    print("\nPhysical Parameters:")
    print("- h: Convective heat transfer coefficient")
    print("- k: Thermal conductivity")
    print("- T∞: Ambient temperature")
    print("- h/k: Biot number (dimensionless)")
    
    print("\nSpecial Cases:")
    print("- h → 0: Neumann BC (insulated)")
    print("- h → ∞: Dirichlet BC (fixed temperature)")
    print("- k → ∞: Perfect conductor")

robin_implementation()

## 5. Complex Geometries: Boundary Detection and Application

Real-world problems rarely involve simple rectangles. Let's explore how to handle boundary conditions on complex geometries.

In [ ]:
def complex_geometry_bcs():
    """
    Apply boundary conditions on complex geometries.
    """
    def create_l_shaped_domain():
        """
        Create mesh for L-shaped domain.
        """
        # Generate points for L-shape
        points = []
        
        # Regular grid
        for i in range(11):
            for j in range(11):
                x = i / 10.0
                y = j / 10.0
                
                # Only keep points in L-shape
                if (x <= 0.6 and y <= 1.0) or (x <= 1.0 and y <= 0.4):
                    points.append([x, y])
        
        # Add some interior points for better mesh
        np.random.seed(42)
        for _ in range(50):
            x = np.random.uniform(0, 1)
            y = np.random.uniform(0, 1)
            
            if (x <= 0.6 and y <= 1.0) or (x <= 1.0 and y <= 0.4):
                if x > 0.05 and x < 0.95 and y > 0.05 and y < 0.95:  # Away from boundaries
                    points.append([x, y])
        
        points = np.array(points)
        tri = Delaunay(points)
        
        return points, tri.simplices
    
    def identify_boundary_types(nodes, elements):
        """
        Identify different types of boundaries for complex geometry.
        """
        boundary_info = {
            'dirichlet': [],
            'neumann': [],
            'robin': []
        }
        
        # Find boundary edges
        edge_count = {}
        edges = []
        
        for elem in elements:
            for i in range(3):
                j = (i + 1) % 3
                edge = tuple(sorted([elem[i], elem[j]]))
                edges.append(edge)
                edge_count[edge] = edge_count.get(edge, 0) + 1
        
        boundary_edges = [edge for edge in edges if edge_count[edge] == 1]
        
        # Classify boundary edges based on location
        for edge in boundary_edges:
            node1, node2 = edge
            x1, y1 = nodes[node1]
            x2, y2 = nodes[node2]
            x_mid, y_mid = (x1 + x2) / 2, (y1 + y2) / 2
            
            # Classify based on position
            tolerance = 0.05
            
            # Left boundary: Dirichlet (fixed temperature)
            if x_mid < tolerance:
                boundary_info['dirichlet'].append((edge, 0.0))
            
            # Top boundary: Neumann (insulated)
            elif abs(y_mid - 1.0) < tolerance and x_mid < 0.6:
                boundary_info['neumann'].append((edge, 0.0))
            
            # Right boundary: Robin (convection)
            elif abs(x_mid - 1.0) < tolerance and y_mid < 0.4:
                boundary_info['robin'].append((edge, 0.0))
            
            # Bottom boundary: Neumann
            elif y_mid < tolerance:
                boundary_info['neumann'].append((edge, 0.0))
            
            # Inner corner: Dirichlet
            elif abs(x_mid - 0.6) < tolerance and abs(y_mid - 0.4) < tolerance:
                boundary_info['dirichlet'].append((edge, 50.0))
        
        return boundary_info
    
    def assemble_fea_system(nodes, elements, source_func):
        """
        Assemble basic FEA system.
        """
        n_nodes = len(nodes)
        K = lil_matrix((n_nodes, n_nodes))
        f = np.zeros(n_nodes)
        
        for elem in elements:
            coords = nodes[elem]
            x_coords, y_coords = coords[:, 0], coords[:, 1]
            
            # Element area
            A = 0.5 * abs(np.linalg.det(np.array([
                [1, x_coords[0], y_coords[0]],
                [1, x_coords[1], y_coords[1]],
                [1, x_coords[2], y_coords[2]]
            ])))
            
            # Shape function gradients
            b = np.array([y_coords[1] - y_coords[2], y_coords[2] - y_coords[0], y_coords[0] - y_coords[1]])
            c = np.array([x_coords[2] - x_coords[1], x_coords[0] - x_coords[2], x_coords[1] - x_coords[0]])
            B = np.array([b, c]) / (2 * A)
            
            # Element stiffness
            Ke = A * (B.T @ B)
            
            # Element load
            centroid = np.mean(coords, axis=0)
            source_val = source_func(centroid[0], centroid[1])
            fe = np.ones(3) * source_val * A / 3.0
            
            # Assembly
            for i in range(3):
                for j in range(3):
                    K[elem[i], elem[j]] += Ke[i, j]
                f[elem[i]] += fe[i]
        
        return K.tocsr(), f
    
    def apply_complex_bcs(nodes, elements, K, f, boundary_info):
        """
        Apply mixed boundary conditions on complex geometry.
        """
        K_modified = K.copy()
        f_modified = f.copy()
        
        # Apply Dirichlet BCs
        dirichlet_nodes = set()
        for edge, value in boundary_info['dirichlet']:
            dirichlet_nodes.update(edge)
        
        penalty = 1e10
        for node in dirichlet_nodes:
            K_modified[node, node] += penalty
            # Find the value for this node
            for edge, value in boundary_info['dirichlet']:
                if node in edge:
                    f_modified[node] += penalty * value
                    break
        
        # Apply Neumann BCs
        for edge, flux in boundary_info['neumann']:
            node1, node2 = edge
            x1, y1 = nodes[node1]
            x2, y2 = nodes[node2]
            edge_length = np.sqrt((x2 - x1)**2 + (y2 - y1)**2)
            
            contribution = flux * edge_length / 2
            f_modified[node1] += contribution
            f_modified[node2] += contribution
        
        # Apply Robin BCs
        for edge, ambient_temp in boundary_info['robin']:
            node1, node2 = edge
            x1, y1 = nodes[node1]
            x2, y2 = nodes[node2]
            edge_length = np.sqrt((x2 - x1)**2 + (y2 - y1)**2)
            
            # Robin parameters
            h_conv = 10.0  # Convection coefficient
            k_thermal = 1.0  # Thermal conductivity
            robin_coeff = (h_conv / k_thermal) * edge_length
            
            # Stiffness contribution
            Ke_robin = robin_coeff * np.array([[1/3, 1/6], [1/6, 1/3]])
            # Load contribution
            fe_robin = robin_coeff * ambient_temp * np.array([1.5, 1.5])
            
            local_nodes = [node1, node2]
            for i in range(2):
                for j in range(2):
                    K_modified[local_nodes[i], local_nodes[j]] += Ke_robin[i, j]
                f_modified[local_nodes[i]] += fe_robin[i]
        
        return K_modified, f_modified, dirichlet_nodes
    
    # Create and solve the complex geometry problem
    nodes, elements = create_l_shaped_domain()
    
    def source_function(x, y):
        """
        Heat source function.
        """
        # Concentrated heat source near the inner corner
        if abs(x - 0.3) < 0.2 and abs(y - 0.2) < 0.2:
            return 100.0
        return 10.0
    
    # Assemble system
    K, f = assemble_fea_system(nodes, elements, source_function)
    
    # Identify boundary conditions
    boundary_info = identify_boundary_types(nodes, elements)
    
    # Apply boundary conditions
    K_bc, f_bc, dirichlet_nodes = apply_complex_bcs(nodes, elements, K, f, boundary_info)
    
    # Solve
    u = spsolve(K_bc, f_bc)
    
    # Visualize results
    fig, axes = plt.subplots(1, 2, figsize=(15, 6))
    
    # Plot 1: Mesh and boundary types
    ax1 = axes[0]
    ax1.triplot(nodes[:, 0], nodes[:, 1], elements, 'k-', linewidth=0.3, alpha=0.5)
    
    # Color boundary edges by type
    for edge, value in boundary_info['dirichlet']:
        x1, y1 = nodes[edge[0]]
        x2, y2 = nodes[edge[1]]
        ax1.plot([x1, x2], [y1, y2], 'r-', linewidth=3, label='Dirichlet' if edge == boundary_info['dirichlet'][0][0] else '')
    
    for edge, flux in boundary_info['neumann']:
        x1, y1 = nodes[edge[0]]
        x2, y2 = nodes[edge[1]]
        ax1.plot([x1, x2], [y1, y2], 'b-', linewidth=3, label='Neumann' if edge == boundary_info['neumann'][0][0] else '')
    
    for edge, temp in boundary_info['robin']:
        x1, y1 = nodes[edge[0]]
        x2, y2 = nodes[edge[1]]
        ax1.plot([x1, x2], [y1, y2], 'g-', linewidth=3, label='Robin' if edge == boundary_info['robin'][0][0] else '')
    
    ax1.set_title('L-Shaped Domain with Mixed Boundary Conditions')
    ax1.set_xlabel('x')
    ax1.set_ylabel('y')
    ax1.set_aspect('equal')
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    
    # Plot 2: Solution
    ax2 = axes[1]
    contour = ax2.tricontourf(nodes[:, 0], nodes[:, 1], elements, u, levels=20, cmap='hot')
    plt.colorbar(contour, ax=ax2, label='Temperature')
    ax2.triplot(nodes[:, 0], nodes[:, 1], elements, 'k-', linewidth=0.2, alpha=0.3)
    ax2.set_title('Temperature Distribution')
    ax2.set_xlabel('x')
    ax2.set_ylabel('y')
    ax2.set_aspect('equal')
    
    plt.tight_layout()
    plt.show()
    
    print("Complex Geometry Boundary Conditions:")
    print("=" * 50)
    print(f"\nBoundary Types Applied:")
    print(f"- Dirichlet nodes: {len(dirichlet_nodes)}")
    print(f"- Neumann edges: {len(boundary_info['neumann'])}")
    print(f"- Robin edges: {len(boundary_info['robin'])}")
    
    print("\nChallenges in Complex Geometries:")
    print("1. Boundary edge detection")
    print("2. Boundary classification based on geometry")
    print("3. Mixed boundary conditions on same domain")
    print("4. Corner and interface treatment")
    
    print("\nSolution Strategy:")
    print("- Edge-based boundary detection")
    print("- Geometric classification functions")
    print("- Separate handling for each BC type")
    print("- Proper bookkeeping for mixed conditions")

complex_geometry_bcs()

## 6. Advanced Topics: Periodic and Coupled Boundary Conditions

### Periodic Boundary Conditions

Periodic boundary conditions connect opposite boundaries, often used for:
- Unit cell analysis in materials science
    Flow in periodic structures
    Electromagnetic problems in crystals

Mathematical form: $u(x=0) = u(x=L)$ and possibly derivatives too.

In [ ]:
def periodic_boundary_conditions():
    """
    Implement periodic boundary conditions.
    """
    def create_periodic_mesh():
        """
        Create mesh with periodic boundaries.
        """
        # Create mesh on [0,1] × [0,1]
        nx, ny = 6, 6
        x = np.linspace(0, 1, nx)
        y = np.linspace(0, 1, ny)
        X, Y = np.meshgrid(x, y)
        nodes = np.column_stack([X.ravel(), Y.ravel()])
        tri = Delaunay(nodes)
        elements = tri.simplices
        
        return nodes, elements
    
    def identify_periodic_nodes(nodes):
        """
        Identify corresponding nodes on periodic boundaries.
        """
        tolerance = 1e-6
        
        # Find nodes on each boundary
        left_nodes = []
        right_nodes = []
        bottom_nodes = []
        top_nodes = []
        
        for i, (x, y) in enumerate(nodes):
            if abs(x) < tolerance:
                left_nodes.append((i, y))
            elif abs(x - 1.0) < tolerance:
                right_nodes.append((i, y))
            
            if abs(y) < tolerance:
                bottom_nodes.append((i, x))
            elif abs(y - 1.0) < tolerance:
                top_nodes.append((i, x))
        
        # Match periodic pairs
        periodic_pairs_x = []
        periodic_pairs_y = []
        
        # Match left-right boundaries
        for left_idx, left_y in left_nodes:
            for right_idx, right_y in right_nodes:
                if abs(left_y - right_y) < tolerance:
                    periodic_pairs_x.append((left_idx, right_idx))
                    break
        
        # Match bottom-top boundaries
        for bottom_idx, bottom_x in bottom_nodes:
            for top_idx, top_x in top_nodes:
                if abs(bottom_x - top_x) < tolerance:
                    periodic_pairs_y.append((bottom_idx, top_idx))
                    break
        
        return periodic_pairs_x, periodic_pairs_y
    
    def apply_periodic_bcs(K, f, periodic_pairs_x, periodic_pairs_y):
        """
        Apply periodic boundary conditions using constraint method.
        """
        n_nodes = K.shape[0]
        K_periodic = K.copy()
        f_periodic = f.copy()
        
        # Keep track of eliminated nodes
        eliminated_nodes = set()
        
        # Apply x-direction periodicity
        for node1, node2 in periodic_pairs_x:
            if node1 not in eliminated_nodes and node2 not in eliminated_nodes:
                # Eliminate node2, keep node1
                eliminated_nodes.add(node2)
                
                # Add contributions from node2 to node1
                K_periodic[node1, :] += K[node2, :]
                f_periodic[node1] += f[node2]
                
                # Add contributions to node1 from other nodes referencing node2
                for i in range(n_nodes):
                    if i != node1 and i != node2:
                        K_periodic[i, node1] += K[i, node2]
                        K_periodic[i, node2] = 0
                
                # Zero out row and column for eliminated node
                K_periodic[node2, :] = 0
                K_periodic[:, node2] = 0
                f_periodic[node2] = 0
        
        # Apply y-direction periodicity
        for node1, node2 in periodic_pairs_y:
            if node1 not in eliminated_nodes and node2 not in eliminated_nodes:
                eliminated_nodes.add(node2)
                
                K_periodic[node1, :] += K[node2, :]
                f_periodic[node1] += f[node2]
                
                for i in range(n_nodes):
                    if i != node1 and i != node2:
                        K_periodic[i, node1] += K[i, node2]
                        K_periodic[i, node2] = 0
                
                K_periodic[node2, :] = 0
                K_periodic[:, node2] = 0
                f_periodic[node2] = 0
        
        # Apply constraint to maintain solution (fix one node)
        reference_node = 0
        penalty = 1e10
        K_periodic[reference_node, reference_node] += penalty
        
        return K_periodic, f_periodic, eliminated_nodes
    
    # Create and solve periodic problem
    nodes, elements = create_periodic_mesh()
    
    # Assemble basic system (Poisson equation)
    n_nodes = len(nodes)
    K = lil_matrix((n_nodes, n_nodes))
    f = np.zeros(n_nodes)
    
    for elem in elements:
        coords = nodes[elem]
        x_coords, y_coords = coords[:, 0], coords[:, 1]
        
        A = 0.5 * abs(np.linalg.det(np.array([
            [1, x_coords[0], y_coords[0]],
            [1, x_coords[1], y_coords[1]],
            [1, x_coords[2], y_coords[2]]
        ])))
        
        b = np.array([y_coords[1] - y_coords[2], y_coords[2] - y_coords[0], y_coords[0] - y_coords[1]])
        c = np.array([x_coords[2] - x_coords[1], x_coords[0] - x_coords[2], x_coords[1] - x_coords[0]])
        B = np.array([b, c]) / (2 * A)
        
        Ke = A * (B.T @ B)
        
        # Add a source term that creates periodic pattern
        centroid = np.mean(coords, axis=0)
        source_val = 10 * np.sin(2 * np.pi * centroid[0]) * np.cos(2 * np.pi * centroid[1])
        fe = np.ones(3) * source_val * A / 3.0
        
        for i in range(3):
            for j in range(3):
                K[elem[i], elem[j]] += Ke[i, j]
            f[elem[i]] += fe[i]
    
    # Apply periodic boundary conditions
    periodic_pairs_x, periodic_pairs_y = identify_periodic_nodes(nodes)
    K_periodic, f_periodic, eliminated_nodes = apply_periodic_bcs(K.tocsr(), f, periodic_pairs_x, periodic_pairs_y)
    
    # Solve
    u_periodic = spsolve(K_periodic, f_periodic)
    
    # Reconstruct full solution
    u_full = u_periodic.copy()
    for node1, node2 in periodic_pairs_x:
        if node2 in eliminated_nodes:
            u_full[node2] = u_full[node1]
    
    for node1, node2 in periodic_pairs_y:
        if node2 in eliminated_nodes:
            u_full[node2] = u_full[node1]
    
    # Visualize
    fig, axes = plt.subplots(1, 2, figsize=(15, 6))
    
    # Plot mesh with periodic connections
    ax1 = axes[0]
    ax1.triplot(nodes[:, 0], nodes[:, 1], elements, 'k-', linewidth=0.5, alpha=0.7)
    ax1.plot(nodes[:, 0], nodes[:, 1], 'ro', markersize=4)
    
    # Show periodic connections
    for node1, node2 in periodic_pairs_x[:3]:  # Show first few connections
        ax1.annotate('', xy=(nodes[node2, 0], nodes[node2, 1]), 
                   xytext=(nodes[node1, 0], nodes[node1, 1]),
                   arrowprops=dict(arrowstyle='<->', color='red', lw=2, alpha=0.7))
    
    for node1, node2 in periodic_pairs_y[:3]:
        ax1.annotate('', xy=(nodes[node2, 0], nodes[node2, 1]), 
                   xytext=(nodes[node1, 0], nodes[node1, 1]),
                   arrowprops=dict(arrowstyle='<->', color='blue', lw=2, alpha=0.7))
    
    ax1.set_title('Periodic Boundary Connections\n(Red: x-periodicity, Blue: y-periodicity)')
    ax1.set_xlabel('x')
    ax1.set_ylabel('y')
    ax1.set_aspect('equal')
    
    # Plot solution
    ax2 = axes[1]
    contour = ax2.tricontourf(nodes[:, 0], nodes[:, 1], elements, u_full, levels=20, cmap='twilight')
    plt.colorbar(contour, ax=ax2)
    ax2.triplot(nodes[:, 0], nodes[:, 1], elements, 'k-', linewidth=0.2, alpha=0.3)
    ax2.set_title('Periodic Solution')
    ax2.set_xlabel('x')
    ax2.set_ylabel('y')
    ax2.set_aspect('equal')
    
    plt.tight_layout()
    plt.show()
    
    print("Periodic Boundary Conditions:")
    print("=" * 40)
    print(f"\nPeriodic pairs identified:")
    print(f"- X-direction: {len(periodic_pairs_x)} pairs")
    print(f"- Y-direction: {len(periodic_pairs_y)} pairs")
    print(f"- Total eliminated nodes: {len(eliminated_nodes)}")
    
    print("\nImplementation Method:")
    print("1. Identify corresponding nodes on periodic boundaries")
    print("2. Eliminate duplicate degrees of freedom")
    print("3. Add contributions from eliminated to retained nodes")
    print("4. Apply reference constraint to prevent singularity")
    
    print("\nApplications:")
    print("- Unit cell analysis in materials science")
    print("- Flow in periodic structures")
    print("- Electromagnetic problems in crystals")
    print("- Turbulence modeling with periodic domains")

periodic_boundary_conditions()

## 7. Key Takeaways

### What We Learned About Boundary Conditions

1. **Mathematical Foundation**: Boundary conditions ensure well-posedness and represent physical reality

2. **Implementation Strategies**: Different BC types require different implementation approaches

3. **Essential vs Natural**: Dirichlet BCs are essential (modify trial space), Neumann BCs are natural (appear in weak form)

4. **Robin BCs**: The most general case, combining value and derivative constraints

5. **Complex Geometries**: Boundary detection and classification are crucial for real-world problems

### Implementation Guidelines

**Dirichlet BCs:**
- Use submatrix method for accuracy
- Penalty method for simplicity
- Always check for well-posedness

**Neumann BCs:**
- Contribute to load vector only
- Use boundary integration
- Require at least one Dirichlet BC

**Robin BCs:**
- Contribute to both stiffness and load
- Model physical convection/interaction
- Bridge between Dirichlet and Neumann

**Complex Geometries:**
- Edge-based boundary detection
- Geometric classification functions
- Careful bookkeeping for mixed BCs

### Physical Interpretation

Boundary conditions are not just mathematical constraints - they represent real physical interactions:

- **Dirichlet**: Prescribed values (temperature, displacement)
- **Neumann**: Prescribed fluxes (heat flow, forces)
- **Robin**: Convective interaction with environment
- **Periodic**: Repeating structures and unit cells

### Common Pitfalls and Solutions

**Pitfall 1**: Insufficient boundary conditions
- **Solution**: Always check for well-posedness
- **Rule**: At least one Dirichlet BC for elliptic problems

**Pitfall 2**: Overconstrained problems
- **Solution**: Check for conflicting BCs
- **Rule**: Ensure consistency at corners and interfaces

**Pitfall 3**: Poor boundary condition implementation
- **Solution**: Use appropriate methods for each BC type
- **Rule**: Test on simple problems with known solutions

**Pitfall 4**: Incorrect boundary identification
- **Solution**: Use robust edge detection algorithms
- **Rule**: Verify boundary classification visually

### Next Steps

This foundation in boundary conditions prepares you for:
- Dynamic problems (time-dependent BCs)
- Nonlinear boundary conditions
- Coupled multi-physics problems
- Contact problems with inequality constraints
- Adaptive boundary condition strategies

Remember: **Boundary conditions are as important as the governing equations themselves!**

The mathematical rigor, physical intuition, and practical implementation skills you've developed here will help you solve real-world engineering problems with confidence and accuracy.